# Qwen3-1.7B on JudgeBench: HPC Smoke Test

This notebook evaluates Qwen3-1.7B as an LLM judge on the local JudgeBench GPT dataset using an SJSU HPC GPU.

The first HPC test uses 4 examples:
- 2 math examples
- 2 code examples

Each example is evaluated in the original and reversed response order.

## HPC execution notes

The model and dataset are already downloaded on HPC.

Model:
`/home/019111677/models/Qwen3-1.7B`

Dataset:
`/home/019111677/datasets/JudgeBench/data/gpt-00000-of-00001.jsonl`

Do not run package-installation, Google Colab, Google Drive, or model-download cells.

In [ ]:
import os
import sys
import json
import re
import time
import random
import pandas as pd
import torch

from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm

# Use personal packages before HPC-wide packages
sys.path.insert(
    0,
    "/home/019111677/venvs/judgebench-gpu/lib/python3.11/site-packages"
)

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["CUDA_HOME"] = (
    "/fs/sfs/local/opt/ohpc/pub/apps/nvidia/"
    "nvhpc/24.11/Linux_x86_64/24.11/compilers"
)
os.environ["TRITON_CACHE_DIR"] = f"/tmp/{os.environ.get('USER', 'user')}/triton"

Path(os.environ["TRITON_CACHE_DIR"]).mkdir(
    parents=True,
    exist_ok=True
)

DEVICE = "cuda:0"
MODEL_NAME = "Qwen3-1.7B"
PRECISION = "float16"
RUN_LIMIT = 4

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available.")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
    "GB"
)

RESULT_DIR = Path(
    "/home/019111677/judgebench/results"
)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Results directory:", RESULT_DIR)

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 15.64 GB
Mounted at /content/drive
Results directory: /content/drive/MyDrive/llm-judge-project/results


In [ ]:
# Load and inspect JudgeBench.
DATA_PATH = (
    "/home/019111677/datasets/JudgeBench/"
    "data/gpt-00000-of-00001.jsonl"
)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Examples:", len(records))
print("Columns:", records[0].keys())

README.md:   0%|          | 0.00/2.70k [00:00<?, ?B/s]

claude-00000-of-00001.jsonl:   0%|          | 0.00/920k [00:00<?, ?B/s]

gpt-00000-of-00001.jsonl:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Generating claude split:   0%|          | 0/270 [00:00<?, ? examples/s]

Generating gpt split:   0%|          | 0/350 [00:00<?, ? examples/s]

Dataset({
    features: ['pair_id', 'original_id', 'source', 'question', 'response_model', 'response_A', 'response_B', 'label'],
    num_rows: 350
})
Columns: ['pair_id', 'original_id', 'source', 'question', 'response_model', 'response_A', 'response_B', 'label']
Total examples: 350
Labels: Counter({'A>B': 193, 'B>A': 157})
Sources: Counter({'livebench-reasoning': 98, 'livebench-math': 56, 'livecodebench': 42, 'mmlu-pro-law': 11, 'mmlu-pro-biology': 11, 'mmlu-pro-computer science': 11, 'mmlu-pro-health': 11, 'mmlu-pro-history': 11, 'mmlu-pro-psychology': 11, 'mmlu-pro-other': 11, 'mmlu-pro-philosophy': 11, 'mmlu-pro-economics': 11, 'mmlu-pro-math': 11, 'mmlu-pro-chemistry': 11, 'mmlu-pro-physics': 11, 'mmlu-pro-engineering': 11, 'mmlu-pro-business': 11})


In [ ]:
math_records = [
    dict(r, task_type="math")
    for r in records
    if r["source"] in {"livebench-math", "mmlu-pro-math"}
]

code_records = [
    dict(r, task_type="code")
    for r in records
    if r["source"] == "livecodebench"
]

random.seed(42)

evaluation_dataset = (
    random.sample(math_records, 2)
    + random.sample(code_records, 2)
)

print("Evaluation size:", len(evaluation_dataset))

for i, record in enumerate(evaluation_dataset):
    print(
        i,
        "| task:", record["task_type"],
        "| source:", record["source"],
        "| label:", record["label"]
    )

Filter:   0%|          | 0/350 [00:00<?, ? examples/s]

Filter:   0%|          | 0/350 [00:00<?, ? examples/s]

Math examples: 67
Code examples: 42
Math sources: Counter({'livebench-math': 56, 'mmlu-pro-math': 11})
Code sources: Counter({'livecodebench': 42})


Flattening the indices:   0%|          | 0/10 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/10 [00:00<?, ? examples/s]

Final evaluation size: 20
Task distribution: Counter({'code': 10, 'math': 10})
Source distribution: Counter({'livecodebench': 10, 'livebench-math': 9, 'mmlu-pro-math': 1})


## Model configuration

We use non-thinking mode so that the judge produces a short, consistent A/B decision. The raw output is still saved for manual inspection.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = "/home/019111677/models/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map={"": 0},
    local_files_only=True
)

print("Model device:", next(model.parameters()).device)

Loading model: Qwen/Qwen3-1.7B
Precision: fp16


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Model device: cuda:0


In [ ]:
def build_prompt(question, response_a, response_b):
    return f"""You are an impartial evaluator.

Question:
{question}

Response A:
{response_a}

Response B:
{response_b}

Which response is more correct?

Return exactly one character:
A
or
B

Do not provide an explanation."""

def expected_choice(label):
    if label == "A>B":
        return "A"

    if label == "B>A":
        return "B"

    return "INVALID"

def parse_choice(raw_output):
    text = raw_output.strip().upper()

    # Prefer an explicit final answer
    explicit = re.findall(
        r"(?:FINAL ANSWER|ANSWER|CHOICE|WINNER)\s*[:\-]?\s*([AB])\b",
        text
    )

    if explicit:
        return explicit[-1]

    # Accept an output consisting essentially of A or B
    if re.fullmatch(r"[\s\[\]():\-]*A[\s\[\]():\-.]*", text):
        return "A"

    if re.fullmatch(r"[\s\[\]():\-]*B[\s\[\]():\-.]*", text):
        return "B"

    # Fallback only if there is exactly one standalone A/B
    matches = re.findall(r"\b([AB])\b", text)

    if len(matches) == 1:
        return matches[0]

    return "INVALID"

In [ ]:
def run_judge(question, response_a, response_b):
    prompt = build_prompt(question, response_a, response_b)

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=8192
    ).to(DEVICE)

    start = time.perf_counter()

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    latency = time.perf_counter() - start

    new_tokens = generated_ids[0][inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return {
        "prompt": prompt,
        "raw_output": raw_output,
        "prediction": parse_choice(raw_output),
        "latency_seconds": latency
    }

In [ ]:
def convert_reversed_choice(choice):
    if choice == "A":
        return "B"
    if choice == "B":
        return "A"
    return "INVALID"


def evaluate_pair(item):
    # gold = "A" if item["label"] == "A>B" else "B"
    gold = expected_choice(item["label"])

    # Original order: A, B
    original = run_judge(
        item["question"],
        item["response_A"],
        item["response_B"]
    )

    # Reversed order: B, A
    reversed_result = run_judge(
        item["question"],
        item["response_B"],
        item["response_A"]
    )

    reversed_original_identity = convert_reversed_choice(
        reversed_result["prediction"]
    )

    return {
        "pair_id": item["pair_id"],
        "task_type": item["task_type"],
        "source": item["source"],
        "label": item["label"],
        "gold": gold,

        "model": MODEL_NAME,
        "precision": PRECISION,

        "original_prediction": original["prediction"],
        "reversed_prediction_presented": reversed_result["prediction"],
        "reversed_prediction_original_identity": reversed_original_identity,

        "original_correct": original["prediction"] == gold,
        "reversed_correct": reversed_original_identity == gold,

        "order_consistent": (
            original["prediction"] != "INVALID"
            and reversed_original_identity != "INVALID"
            and original["prediction"] == reversed_original_identity
        ),

        "original_raw_output": original["raw_output"],
        "reversed_raw_output": reversed_result["raw_output"],

        "original_latency_seconds": original["latency_seconds"],
        "reversed_latency_seconds": reversed_result["latency_seconds"],
    }

In [ ]:
# RUN_NAME = ("qwen3_1.7b_judgebench_gpt_mathcode_pilot")
RUN_NAME = (
    "qwen3_1.7b_judgebench_hpc_smoke_4"
)

JSONL_PATH = RESULT_DIR / f"{RUN_NAME}.jsonl"
CSV_PATH = RESULT_DIR / f"{RUN_NAME}.csv"

completed_ids = set()

if JSONL_PATH.exists():
    with JSONL_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)
                completed_ids.add(record["pair_id"])
            except Exception:
                pass

print("Already completed:", len(completed_ids))

new_records = []

for item in tqdm(evaluation_dataset):
    if item["pair_id"] in completed_ids:
        continue

    record = evaluate_pair(item)
    new_records.append(record)

    # Save after every example.
    with JSONL_PATH.open("a", encoding="utf-8") as f:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

print("New records:", len(new_records))
print("Saved to:", JSONL_PATH)

Already completed: 0


  0%|          | 0/20 [00:00<?, ?it/s]

New records: 20
Saved to: /content/drive/MyDrive/llm-judge-project/results/qwen3_1.7b_judgebench_gpt_balanced_math_code_20.jsonl


In [ ]:
result_records = []

with JSONL_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        result_records.append(json.loads(line))

results = pd.DataFrame(result_records)

def calculate_metrics(df):
    original_valid = (
        df["original_prediction"] != "INVALID"
    )

    reversed_valid = (
        df["reversed_prediction_presented"] != "INVALID"
    )

    both_valid = original_valid & reversed_valid

    return {
        "examples": len(df),

        "original_accuracy_all": (
            df["original_correct"].mean()
        ),

        "reversed_accuracy_all": (
            df["reversed_correct"].mean()
        ),

        "original_accuracy_valid_only": (
            df.loc[
                original_valid,
                "original_correct"
            ].mean()
        ),

        "reversed_accuracy_valid_only": (
            df.loc[
                reversed_valid,
                "reversed_correct"
            ].mean()
        ),

        "original_invalid_rate": (
            (~original_valid).mean()
        ),

        "reversed_invalid_rate": (
            (~reversed_valid).mean()
        ),

        "order_consistency_all": (
            df["order_consistent"].mean()
        ),

        "order_consistency_valid_only": (
            df.loc[
                both_valid,
                "order_consistent"
            ].mean()
        ),

        "average_latency_seconds": (
            df[
                [
                    "original_latency_seconds",
                    "reversed_latency_seconds"
                ]
            ].mean().mean()
        )
    }

overall_metrics = calculate_metrics(results)

print("Overall results:")
display(pd.Series(overall_metrics))

print("Results by task:")
task_metrics = []

for task_type, group in results.groupby("task_type"):
    metrics = calculate_metrics(group)
    metrics["task_type"] = task_type
    task_metrics.append(metrics)

task_metrics_df = pd.DataFrame(task_metrics)
display(task_metrics_df)


results.to_csv(CSV_PATH, index=False)

summary_path = RESULT_DIR / f"{RUN_NAME}_summary.json"

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "overall": overall_metrics,
            "by_task": task_metrics,
        },
        f,
        indent=2
    )

print("CSV saved to:", CSV_PATH)
print("Summary saved to:", summary_path)

Overall results:


,0
examples,20.000000
original_accuracy_all,0.500000
reversed_accuracy_all,0.400000
original_accuracy_valid_only,0.588235
reversed_accuracy_valid_only,0.444444
original_invalid_rate,0.150000
reversed_invalid_rate,0.100000
order_consistency_all,0.000000
order_consistency_valid_only,0.000000
average_latency_seconds,0.927284


Results by task:


,examples,original_accuracy_all,reversed_accuracy_all,original_accuracy_valid_only,reversed_accuracy_valid_only,original_invalid_rate,reversed_invalid_rate,order_consistency_all,order_consistency_valid_only,average_latency_seconds,task_type
0,10,0.4,0.5,0.444444,0.555556,0.1,0.1,0.0,0.0,0.524851,code
1,10,0.6,0.3,0.750000,0.333333,0.2,0.1,0.0,0.0,1.329718,math


CSV saved to: /content/drive/MyDrive/llm-judge-project/results/qwen3_1.7b_judgebench_gpt_balanced_math_code_20.csv
Summary saved to: /content/drive/MyDrive/llm-judge-project/results/qwen3_1.7b_judgebench_gpt_balanced_math_code_20_summary.json


## References

1. Tan et al. (2024), **JudgeBench: A Benchmark for Evaluating LLM-Based Judges**: https://arxiv.org/abs/2410.12784
2. Yang et al. (2025), **Qwen3 Technical Report**: https://arxiv.org/abs/2505.09388
3. JudgeBench dataset: https://huggingface.co/datasets/ScalerLab/JudgeBench
4. Qwen3-1.7B model: https://huggingface.co/Qwen/Qwen3-1.7B

## HPC smoke-test status

This notebook is being migrated from Colab to the SJSU HPC.

The HPC environment has been verified:
- Python 3.11.13
- PyTorch 2.6.0 with CUDA 12.4
- Qwen3-1.7B loaded locally
- GPU inference successfully tested
- JudgeBench GPT split downloaded locally

The first HPC evaluation uses 4 examples:
- 2 math examples
- 2 code examples
- Original response order
- Reversed response order

Full evaluation results will be added after the smoke test succeeds.